In [2]:
# --- BLOQUE INICIAL ---
import os
import xarray as xr
import geopandas as gpd
import rioxarray  # si queremos recortar usando geometrías shapely
import pickle

In [3]:
# Paths relativos
BASE_DIR = os.path.abspath("..")
DATA_RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
DATA_PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)

In [4]:
# Archivo NetCDF original de pH
ph_file = os.path.join(DATA_RAW_DIR, "cmems_obs-mob_glo_bgc-car_my_irr-i_1756152669355.nc")

In [5]:
# Cargar NetCDF
ds_ph = xr.open_dataset(ph_file)
print(ds_ph)

<xarray.Dataset> Size: 12MB
Dimensions:               (time: 468, latitude: 14, longitude: 37)
Coordinates:
  * time                  (time) datetime64[ns] 4kB 1985-01-01 ... 2023-12-01
  * latitude              (latitude) float32 56B -55.88 -55.62 ... -52.88 -52.62
  * longitude             (longitude) float32 148B -63.12 -62.88 ... -54.12
Data variables:
    omega_ar              (time, latitude, longitude) float32 970kB ...
    omega_ar_uncertainty  (time, latitude, longitude) float32 970kB ...
    omega_ca              (time, latitude, longitude) float32 970kB ...
    omega_ca_uncertainty  (time, latitude, longitude) float32 970kB ...
    ph                    (time, latitude, longitude) float32 970kB ...
    ph_uncertainty        (time, latitude, longitude) float32 970kB ...
    spco2                 (time, latitude, longitude) float32 970kB ...
    spco2_uncertainty     (time, latitude, longitude) float32 970kB ...
    talk                  (time, latitude, longitude) float32 970

In [11]:
ds_ph = ds_ph.rio.write_crs("EPSG:4326")  # aseguramos CRS


In [12]:
# Cargar polígonos procesados
polygons_file = os.path.join(DATA_PROCESSED_DIR, "bbi_bbii_polygons.pkl")
with open(polygons_file, "rb") as f:
    polygons = pickle.load(f)
bbi_polygon = polygons["BBI"]
bbii_polygon = polygons["BBII"]

In [13]:
# --- RECORTE POR ÁREA ---
# Crear diccionario para almacenar datasets por área
ds_dict = {}

In [14]:
for name, polygon in zip(["BBI", "BBII"], [bbi_polygon, bbii_polygon]):
    ds_area = ds_ph.rio.clip([polygon], all_touched=True, drop=False)
    ds_dict[name] = ds_area

In [15]:
# --- CREAR NETCDF UNIFICADO ---
# Concatenar a lo largo de una dimensión 'area'
ds_combined = xr.concat([ds_dict["BBI"], ds_dict["BBII"]], dim="area")
ds_combined = ds_combined.assign_coords(area=["BBI", "BBII"])

In [16]:
# --- GUARDAR NETCDF PROCESADO ---
outfile = os.path.join(DATA_PROCESSED_DIR, "bbi_bbii_ph.nc")
ds_combined.to_netcdf(outfile)
print(f"NetCDF único guardado en: {outfile}")

NetCDF único guardado en: C:\Users\gisel\BB_stress_paper\data\processed\bbi_bbii_ph.nc
